In [11]:
import os

print(os.getcwd())
print(os.listdir("../backend/data/raw"))

d:\HaryanaSarthi\notebooks
['government_colleges_haryana.csv', 'HaryanaScheme.xlsx', 'Haryana_Government_Internships_Dataset.xlsx', 'Haryana_Govt_Colleges_100.xlsx', 'haryana_job_exam.xlsx', 'India_Government_Scholarships_25k_Dataset.xlsx', 'Insurer-wise Persons Covered under Govt Schemes .xlsx', 'Progress_of_OAS-Allowance_Widow-Pensn_Disabled-Pensn_and_Ladli-SS-Schemes_Haryana_2013-14.xlsx', 'SchemesYear-wise Employment Opportunities.xlsx', 'Unemployment in India.xlsx']


In [12]:
import pandas as pd

jobs_df = pd.read_excel("../backend/data/raw/haryana_job_exam.xlsx")

jobs_df.info()
jobs_df.isnull().sum()
jobs_df.duplicated().sum()

<class 'pandas.DataFrame'>
RangeIndex: 1048575 entries, 0 to 1048574
Data columns (total 22 columns):
 #   Column                   Non-Null Count    Dtype  
---  ------                   --------------    -----  
 0   exam_id                  50000 non-null    float64
 1   exam_name                50000 non-null    str    
 2   post_name                50000 non-null    str    
 3   department               50000 non-null    str    
 4   exam_category            50000 non-null    str    
 5   min_age                  50000 non-null    float64
 6   max_age                  50000 non-null    float64
 7   education_required       50000 non-null    str    
 8   attempts_limit           50000 non-null    str    
 9   state                    50000 non-null    str    
 10  job_location             50000 non-null    str    
 11  apply_link_job           1048575 non-null  str    
 12  candidate_category       50000 non-null    str    
 13  age_relaxation           50000 non-null    float64
 1

np.int64(998574)

In [13]:
jobs_df[['min_age','max_age']].describe()

,min_age,max_age
count,50000.000000,50000.000000
mean,19.492620,33.487080
std,1.117538,4.030002
min,18.000000,27.000000
25%,18.000000,30.000000
50%,19.000000,33.000000
75%,20.000000,37.000000
max,21.000000,40.000000


In [14]:
(jobs_df['min_age'] > jobs_df['max_age']).sum()

np.int64(0)

In [15]:
jobs_df['eligibility_status'].unique()

<StringArray>
['Recommended', 'Highly Recommended', 'Partially Eligible', nan]
Length: 4, dtype: str

In [16]:
jobs_df['eligibility_status'] = jobs_df['eligibility_status'].str.lower().str.strip()

In [17]:
jobs_df['eligibility_status'].unique()

<StringArray>
['recommended', 'highly recommended', 'partially eligible', nan]
Length: 4, dtype: str

In [18]:
jobs_df['candidate_category'].unique()

<StringArray>
['General', 'SC', 'ST', 'OBC', 'EWS', nan]
Length: 6, dtype: str

In [19]:
jobs_df['attempts_limit'].unique()

<StringArray>
['Limited', 'Unlimited', nan]
Length: 3, dtype: str

In [20]:
jobs_df['attempts_limit'] = jobs_df['attempts_limit'].astype(str).str.lower().str.strip()

In [21]:
# unlimited flag
jobs_df['attempts_unlimited'] = jobs_df['attempts_limit'].isin(['no limit', 'unlimited'])

In [22]:
jobs_df['attempts_limit_num'] = jobs_df['attempts_limit'].str.extract(r'(\d+)')
jobs_df['attempts_limit_num'] = pd.to_numeric(jobs_df['attempts_limit_num'], errors='coerce')

jobs_df.loc[jobs_df['attempts_limit'] == 'varies', 'attempts_limit_num'] = np.nan

In [23]:
jobs_df[['attempts_limit','attempts_unlimited','attempts_limit_num']].head(10)
jobs_df['attempts_unlimited'].value_counts()
jobs_df['attempts_limit_num'].value_counts(dropna=False)

attempts_limit_num
NaN    1048575
Name: count, dtype: int64

In [24]:
jobs_df.loc[jobs_df['attempts_unlimited'] == True, 'attempts_limit_num'] = 999

In [25]:
jobs_df['attempts_limit_num'].value_counts()

attempts_limit_num
999.0    25014
Name: count, dtype: int64

In [26]:
jobs_df['candidate_category'] = jobs_df['candidate_category'].str.lower().str.strip()

In [27]:
jobs_df['candidate_category'].unique()

<StringArray>
['general', 'sc', 'st', 'obc', 'ews', nan]
Length: 6, dtype: str

In [28]:
jobs_df['education_required'] = jobs_df['education_required'].str.lower().str.strip()
jobs_df['education_required'].value_counts()

education_required
graduate              10063
post graduate         10057
12th pass             10018
engineering degree     9993
10th pass              9869
Name: count, dtype: int64

In [29]:
jobs_df['state'] = jobs_df['state'].str.lower().str.strip()

jobs_df['is_haryana_job'] = jobs_df['state'].isin(['haryana', 'all india'])

In [30]:
jobs_df['is_haryana_job'].value_counts()

is_haryana_job
False    998575
True      50000
Name: count, dtype: int64

In [31]:
jobs_df['is_haryana_eligible_job'].value_counts()

is_haryana_eligible_job
1.0    25144
0.0    24856
Name: count, dtype: int64

In [32]:
import re
import numpy as np

# 1️⃣ Normalize text columns
jobs_df['state'] = jobs_df['state'].astype(str).str.lower().str.strip()
jobs_df['exam_name'] = jobs_df['exam_name'].astype(str).str.lower().str.strip()
jobs_df['exam_category'] = jobs_df['exam_category'].astype(str).str.lower().str.strip()

# 2️⃣ National recruitment body keywords (tight & safe list)
national_keywords = [
    r'\bssc\b',
    r'\bupsc\b',
    r'\brrb\b',
    r'\brailway\b',
    r'\bibps\b',
    r'\bsbi\b',
    r'\bafcat\b'
]

pattern = re.compile("|".join(national_keywords), flags=re.IGNORECASE)

# 3️⃣ Detect national exams
jobs_df['is_national_exam'] = (
    jobs_df['exam_name'].str.contains(pattern, na=False) |
    jobs_df['exam_category'].str.contains(pattern, na=False)
)

# 4️⃣ Final Haryana eligibility logic
jobs_df['is_haryana_eligible_job'] = (
    jobs_df['state'].isin(['haryana', 'all india']) |
    jobs_df['is_national_exam'] |
    (jobs_df['exam_category'] == 'central')
)

# 5️⃣ Extract Haryana-eligible jobs
haryana_jobs_df = jobs_df[jobs_df['is_haryana_eligible_job']].copy()

print("Total jobs:", len(jobs_df))
print("Haryana-eligible jobs:", len(haryana_jobs_df))

# 6️⃣ Save final dataset
haryana_jobs_df.to_csv("../data/cleaned/haryana_jobs_final.csv", index=False)

print("✅ Saved: Jobs&Exams_cleaned.csv")

Total jobs: 1048575
Haryana-eligible jobs: 50000


OSError: Cannot save file into a non-existent directory: '..\data\cleaned'